## rMATS significant events: key genomic coordinates (BED)

This notebook reads rMATS JCEC event files (SE/A5SS/A3SS/RI/MXE), filters significant events, and outputs key genomic coordinates in BED format.

How coordinates are derived (rMATS column reference; starts with `_0base` are already 0-based):
- SE (Skipped Exon): use the skipped exon region `[exonStart_0base, exonEnd)`.
- RI (Retained Intron): use the retained intron region `[riExonStart_0base, riExonEnd)`.
- A3SS (Alt 3' splice site): output two segments: the short acceptor segment `[shortES, shortEE)` and the long segment `[longExonStart_0base, longExonEnd)`.
- A5SS (Alt 5' splice site): output two segments: the short donor segment `[shortES, shortEE)` and the long segment `[longExonStart_0base, longExonEnd)`.
- MXE (Mutually exclusive exons): output both exons `[1stExonStart_0base, 1stExonEnd)` and `[2ndExonStart_0base, 2ndExonEnd)`.

Notes:
- BED is 0-based, half-open; rMATS `*_0base` columns are already 0-based, so we use them directly.
- We also include columns: `name` (event_type|gene|label), `score` (scaled |dPSI| to 0–1000), `strand`, and metadata (FDR, IncLevelDifference, gene_id).



In [1]:
from pathlib import Path

FILES = [
    "/path/to/rMATS/output/SE.MATS.JCEC.txt",
    "/path/to/rMATS/output/A5SS.MATS.JCEC.txt",
    "/path/to/rMATS/output/A3SS.MATS.JCEC.txt",
    "/path/to/rMATS/output/RI.MATS.JCEC.txt",
    "/path/to/rMATS/output/MXE.MATS.JCEC.txt",
]

FDR_THRESHOLD = 0.05
DELTA_PSI_THRESHOLD = 0.1

OUT_DIR = "/path/to/rMATS/output/rmats_coords"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

print("Parameters set.")


Parameters set.


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path


def infer_type_from_name(fp: str) -> str:
    name = Path(fp).name
    for t in ["SE", "A5SS", "A3SS", "RI", "MXE"]:
        if name.startswith(t + ".") or name.startswith(t + "_"):
            return t
    return "NA"


def read_rmats(fp: str) -> pd.DataFrame:
    df = pd.read_csv(fp, sep='\t', dtype=str)
    for c in ["FDR", "IncLevelDifference"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    if 'geneSymbol' in df.columns:
        df['gene_name'] = df['geneSymbol'].astype(str).str.split(',').str[0].str.split(';').str[0].str.strip()
    elif 'GeneSymbol' in df.columns:
        df['gene_name'] = df['GeneSymbol'].astype(str).str.split(',').str[0].str.split(';').str[0].str.strip()
    else:
        df['gene_name'] = ''
    if 'GeneID' in df.columns:
        df['gene_id'] = df['GeneID'].astype(str).str.split(',').str[0].str.split(';').str[0].str.strip()
    elif 'gene_id' in df.columns:
        df['gene_id'] = df['gene_id'].astype(str).str.split(',').str[0].str.split(';').str[0].str.strip()
    else:
        df['gene_id'] = ''
    return df


def score_from_dpsi(dpsi: float) -> int:
    if pd.isna(dpsi):
        return 0
    s = int(round(min(1.0, max(0.0, abs(float(dpsi)))) * 1000))
    return s


def se_to_bed(df: pd.DataFrame) -> pd.DataFrame:
    need = ["chr", "exonStart_0base", "exonEnd", "strand", "FDR", "IncLevelDifference", "gene_name", "gene_id"]
    df2 = df[[c for c in need if c in df.columns]].copy()
    df2 = df2.dropna(subset=["chr", "exonStart_0base", "exonEnd", "strand"]).copy()
    df2['start'] = pd.to_numeric(df2['exonStart_0base'], errors='coerce').astype('Int64')
    df2['end'] = pd.to_numeric(df2['exonEnd'], errors='coerce').astype('Int64')
    df2 = df2.dropna(subset=['start', 'end'])
    df2['name'] = ("SE|" + df2['gene_name'].fillna('') + "|exon")
    df2['score'] = df2['IncLevelDifference'].apply(score_from_dpsi)
    out = df2.rename(columns={"chr":"chrom"})[
        ['chrom','start','end','name','score','strand','gene_name','gene_id','FDR','IncLevelDifference']
    ]
    return out


def ri_to_bed(df: pd.DataFrame) -> pd.DataFrame:
    need = ["chr", "riExonStart_0base", "riExonEnd", "strand", "FDR", "IncLevelDifference", "gene_name", "gene_id"]
    df2 = df[[c for c in need if c in df.columns]].copy()
    df2 = df2.dropna(subset=["chr", "riExonStart_0base", "riExonEnd", "strand"]).copy()
    df2['start'] = pd.to_numeric(df2['riExonStart_0base'], errors='coerce').astype('Int64')
    df2['end'] = pd.to_numeric(df2['riExonEnd'], errors='coerce').astype('Int64')
    df2 = df2.dropna(subset=['start', 'end'])
    df2['name'] = ("RI|" + df2['gene_name'].fillna('') + "|intron")
    df2['score'] = df2['IncLevelDifference'].apply(score_from_dpsi)
    out = df2.rename(columns={"chr":"chrom"})[
        ['chrom','start','end','name','score','strand','gene_name','gene_id','FDR','IncLevelDifference']
    ]
    return out


def a3ss_to_bed(df: pd.DataFrame) -> pd.DataFrame:
    need = ["chr","longExonStart_0base","longExonEnd","shortES","shortEE","strand","FDR","IncLevelDifference","gene_name","gene_id"]
    d = df[[c for c in need if c in df.columns]].copy()
    d = d.dropna(subset=["chr","strand"]).copy()
    rows = []
    for _, r in d.iterrows():
        try:
            chrom = r['chr']
            strand = r['strand']
            fdr = r.get('FDR', np.nan)
            dpsi = r.get('IncLevelDifference', np.nan)
            score = score_from_dpsi(dpsi)
            g = r.get('gene_name','')
            gid = r.get('gene_id','')
            # short segment
            if pd.notna(r.get('shortES')) and pd.notna(r.get('shortEE')):
                s1 = int(float(r['shortES']))
                e1 = int(float(r['shortEE']))
                rows.append([chrom,s1,e1,f"A3SS|{g}|short",score,strand,g,gid,fdr,dpsi])
            # long segment
            if pd.notna(r.get('longExonStart_0base')) and pd.notna(r.get('longExonEnd')):
                s2 = int(float(r['longExonStart_0base']))
                e2 = int(float(r['longExonEnd']))
                rows.append([chrom,s2,e2,f"A3SS|{g}|long",score,strand,g,gid,fdr,dpsi])
        except Exception:
            continue
    return pd.DataFrame(rows, columns=['chrom','start','end','name','score','strand','gene_name','gene_id','FDR','IncLevelDifference'])


def a5ss_to_bed(df: pd.DataFrame) -> pd.DataFrame:
    need = ["chr","longExonStart_0base","longExonEnd","shortES","shortEE","strand","FDR","IncLevelDifference","gene_name","gene_id"]
    d = df[[c for c in need if c in df.columns]].copy()
    d = d.dropna(subset=["chr","strand"]).copy()
    rows = []
    for _, r in d.iterrows():
        try:
            chrom = r['chr']
            strand = r['strand']
            fdr = r.get('FDR', np.nan)
            dpsi = r.get('IncLevelDifference', np.nan)
            score = score_from_dpsi(dpsi)
            g = r.get('gene_name','')
            gid = r.get('gene_id','')
            # short segment
            if pd.notna(r.get('shortES')) and pd.notna(r.get('shortEE')):
                s1 = int(float(r['shortES']))
                e1 = int(float(r['shortEE']))
                rows.append([chrom,s1,e1,f"A5SS|{g}|short",score,strand,g,gid,fdr,dpsi])
            # long segment
            if pd.notna(r.get('longExonStart_0base')) and pd.notna(r.get('longExonEnd')):
                s2 = int(float(r['longExonStart_0base']))
                e2 = int(float(r['longExonEnd']))
                rows.append([chrom,s2,e2,f"A5SS|{g}|long",score,strand,g,gid,fdr,dpsi])
        except Exception:
            continue
    return pd.DataFrame(rows, columns=['chrom','start','end','name','score','strand','gene_name','gene_id','FDR','IncLevelDifference'])


def mxe_to_bed(df: pd.DataFrame) -> pd.DataFrame:
    need = ["chr","1stExonStart_0base","1stExonEnd","2ndExonStart_0base","2ndExonEnd","strand","FDR","IncLevelDifference","gene_name","gene_id"]
    d = df[[c for c in need if c in df.columns]].copy()
    d = d.dropna(subset=["chr","strand"]).copy()
    rows = []
    for _, r in d.iterrows():
        try:
            chrom = r['chr']
            strand = r['strand']
            fdr = r.get('FDR', np.nan)
            dpsi = r.get('IncLevelDifference', np.nan)
            score = score_from_dpsi(dpsi)
            g = r.get('gene_name','')
            gid = r.get('gene_id','')
            s1 = int(float(r['1stExonStart_0base']))
            e1 = int(float(r['1stExonEnd']))
            s2 = int(float(r['2ndExonStart_0base']))
            e2 = int(float(r['2ndExonEnd']))
            rows.append([chrom,s1,e1,f"MXE|{g}|exon1",score,strand,g,gid,fdr,dpsi])
            rows.append([chrom,s2,e2,f"MXE|{g}|exon2",score,strand,g,gid,fdr,dpsi])
        except Exception:
            continue
    return pd.DataFrame(rows, columns=['chrom','start','end','name','score','strand','gene_name','gene_id','FDR','IncLevelDifference'])


print("Utilities loaded.")


Utilities loaded.


In [3]:
import pandas as pd

all_bed = []
for fp in FILES:
    et = infer_type_from_name(fp)
    print(f"Reading {et}: {fp}")
    df = read_rmats(fp)
    before = len(df)
    df_sig = df[(df['FDR'] <= FDR_THRESHOLD) & (df['IncLevelDifference'].abs() >= DELTA_PSI_THRESHOLD)].copy()
    print(f"  rows: {before} -> significant: {len(df_sig)}")

    if et == 'SE':
        bed = se_to_bed(df_sig)
    elif et == 'RI':
        bed = ri_to_bed(df_sig)
    elif et == 'A3SS':
        bed = a3ss_to_bed(df_sig)
    elif et == 'A5SS':
        bed = a5ss_to_bed(df_sig)
    elif et == 'MXE':
        bed = mxe_to_bed(df_sig)
    else:
        bed = pd.DataFrame(columns=['chrom','start','end','name','score','strand','gene_name','gene_id','FDR','IncLevelDifference'])

    bed['event_type'] = et
    all_bed.append(bed)

if all_bed:
    merged = pd.concat(all_bed, ignore_index=True)
else:
    merged = pd.DataFrame(columns=['chrom','start','end','name','score','strand','gene_name','gene_id','FDR','IncLevelDifference','event_type'])

# drop invalid
merged = merged.dropna(subset=['chrom','start','end','strand'])
merged = merged[(merged['end'] > merged['start'])]

# Sort by chrom, start
merged = merged.sort_values(['chrom','start','end']).reset_index(drop=True)

# Save BED (bed6 + extra columns allowed)
out_bed = Path(OUT_DIR) / 'rmats_significant_key_coords.bed'
merged.to_csv(out_bed, sep='\t', header=False, index=False, columns=['chrom','start','end','name','score','strand','gene_name','gene_id','FDR','IncLevelDifference','event_type'])
print('Saved BED:', out_bed)

print('\nPreview (top 20 lines):')
print(merged.head(20).to_string(index=False))


Reading SE: /path/to/rMATS/output/SE.MATS.JCEC.txt
  rows: 27160 -> significant: 693
Reading A5SS: /path/to/rMATS/output/A5SS.MATS.JCEC.txt
  rows: 652 -> significant: 38
Reading A3SS: /path/to/rMATS/output/A3SS.MATS.JCEC.txt
  rows: 1103 -> significant: 45
Reading RI: /path/to/rMATS/output/RI.MATS.JCEC.txt
  rows: 978 -> significant: 87
Reading MXE: /path/to/rMATS/output/MXE.MATS.JCEC.txt
  rows: 4579 -> significant: 123
Saved BED: /path/to/rMATS/output/rmats_coords/rmats_significant_key_coords.bed

Preview (top 20 lines):
chrom     start       end                name  score strand gene_name             gene_id           FDR  IncLevelDifference event_type
 chr1    517095    517172      SE|ERMARD|exon    159      -    ERMARD  ENSSSCG00000004010  1.183095e-02               0.159         SE
 chr1    527369    527438    MXE|ERMARD|exon1    128      -    ERMARD  ENSSSCG00000004010  4.949118e-02              -0.128        MXE
 chr1    528442    528472    MXE|ERMARD|exon2    128      -    ER